The infamus way to divide optimizers, gradients, and parameters across GPUs.

ZeRO progressively eliminates redundancies:

- ZeRO-1: Partitions optimizer states
- ZeRO-2: Partitions gradients + optimizer states
- ZeRO-3: Partitions parameters + gradients + optimizer states

We'll show them all together in this notebook, and then add data parallel on top of it.


## ZeRO 1

Say we have a simple little model

In [8]:
import torch
import torch.nn as nn
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 3)
        self.layer2 = nn.Linear(3, 2)
    
    def forward(self, x):
        return self.layer2(torch.relu(self.layer1(x)))

In [9]:
model = TinyModel()

x = torch.randn(1, 4)
print(f"Input:  {x.shape}")
print(f"Output: {model(x).shape}")

print("\nParameters:")
for name, p in model.named_parameters():
    print(f"  {name}: {p.shape}")

Input:  torch.Size([1, 4])
Output: torch.Size([1, 2])

Parameters:
  layer1.weight: torch.Size([3, 4])
  layer1.bias: torch.Size([3])
  layer2.weight: torch.Size([2, 3])
  layer2.bias: torch.Size([2])


Now, we understand that ZeRO 1 should somewhat evenly distribute the parameters's gradients, how do we do that? Through a greedy size based partitioning.

1. Sort all params by size (largest first)
2. Assign each param to the rank with smallest current total bytes
3. BUT - they also consider communication efficiency: params accessed together in same layer are kept on same GPU when possible

In [14]:
param_sizes = []
for i, param in enumerate(model.parameters()):
    num_elements = param.numel()  # total number of scalars in tensor
    bytes_per_element = 4  # fp32
    num_states = 2  # m and v
    
    optimizer_state_bytes = num_elements * num_states * bytes_per_element
    
    param_sizes.append((optimizer_state_bytes, i))
    
    print(f"Param {i} {num_elements} elements = {optimizer_state_bytes} bytes")

Param 0 12 elements = 96 bytes
Param 1 3 elements = 24 bytes
Param 2 6 elements = 48 bytes
Param 3 2 elements = 16 bytes


Now, assume we have 2 GPUs, we would assigne them (trying) to make it equally

In [15]:
sorted_param_sizes = sorted(param_sizes, reverse=True)
sorted_param_sizes

[(96, 0), (48, 2), (24, 1), (16, 3)]

In [16]:
world_size = 2  # 2 GPUs

# Initialize bytes per GPU to 0
gpu_bytes = [0] * world_size 

# Assign each param to GPU with smallest current total
assignments = {}  # param_index -> gpu_rank


for size_bytes, param_idx in sorted_param_sizes:  # this is already sorted

    # Find GPU with minimum bytes
    target_gpu = 0 if gpu_bytes[0] <= gpu_bytes[1] else 1
    
    # Assign
    assignments[param_idx] = target_gpu
    gpu_bytes[target_gpu] += size_bytes
    
    print(f"Param {param_idx} ({size_bytes}B) -> GPU {target_gpu}, GPU{target_gpu} now has {gpu_bytes[target_gpu]}B")
print(f"\nFinal: GPU0 = {gpu_bytes[0]}B, GPU1 = {gpu_bytes[1]}B")

Param 0 (96B) -> GPU 0, GPU0 now has 96B
Param 2 (48B) -> GPU 1, GPU1 now has 48B
Param 1 (24B) -> GPU 1, GPU1 now has 72B
Param 3 (16B) -> GPU 1, GPU1 now has 88B

Final: GPU0 = 96B, GPU1 = 88B


Now that you understand the basics of zero 1, we can implement it as a class!

In [ ]:
class ZeRO1Optimizer:

    def __init__(self, model, rank, world_size, lr=0.001, betas=(0.9, 0.999), eps=1e-8):
        self.rank = rank
        self.world_size = world_size
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.step_count = 0

        self.params = list(model.parameters())

        # Step 1: Calculate optimizer state size for each param
        # Adam stores: momentum (m) + variance (v), both fp32
        param_sizes = []
        for i, param in enumerate(self.params):
            num_elements = param.numel()
            bytes_per_element = 4  # fp32
            num_states = 2  # m and v
            optimizer_state_bytes = num_elements * num_states * bytes_per_element
            param_sizes.append((optimizer_state_bytes, i))

        # Step 2: Sort by size descending (biggest first)
        sorted_param_sizes = sorted(param_sizes, reverse=True)

        # Step 3: Greedy assignment - assign each param to GPU with smallest total
        gpu_bytes = [0] * world_size
        self.owner = {}  # param_idx -> gpu_rank

        for size_bytes, param_idx in sorted_param_sizes:
            target_gpu = gpu_bytes.index(min(gpu_bytes))
            self.owner[param_idx] = target_gpu
            gpu_bytes[target_gpu] += size_bytes

        # Which params does THIS rank own?
        self.owned_indices = [i for i, r in self.owner.items() if r == rank]

        # Create optimizer states ONLY for owned params (fp32)
        self.state = {}
        for i in self.owned_indices:
            param = self.params[i]
            self.state[i] = {
                "m": torch.zeros_like(param, dtype=torch.float32),
                "v": torch.zeros_like(param, dtype=torch.float32),
            }